# 00 · Build the unified dataset

**Shared pipeline notebook (team-owned).** Run it once after `tools/get_data.py`, and again whenever this notebook or the raw data changes. Changes go through a PR.

**Output:** `data/clean/films_unified.csv`, **one row per film** with every source attached. All other notebooks start from this file instead of re-merging. It is gitignored, so each person builds their own copy.

| Source | Adds | Join |
|---|---|---|
| TMDB | Base table: one row per TMDB id | — |
| The Numbers | Budget, **worldwide** gross, MPAA, source, franchise, companies | Title + year matcher (§4), hand-checked 50/50 |
| Wikidata snapshot (committed) | Budget and box office in USD | TMDB id |
| Box Office Mojo | Domestic gross, theaters (US theatrical run) | Title + year matcher |
| RT stefanoleone992 | `actors`, `production_company` | Title + year matcher |
| RT andrezaza | Director, writer, RT box office, ratings | Matt's join rules (§7), hand-checked 0/50 wrong |

**Combined financials:** `fin_budget` / `fin_revenue` take the first available value from TMDB → The Numbers → Wikidata; `fin_*_src` records which. `usable` = 2000–2023, budget ≥ $1M, revenue > 0.

Column prefixes: none = TMDB, `tn_` The Numbers, `wd_` Wikidata, `bom_` Box Office Mojo, `sl_` RT stefanoleone992, `rt_` RT andrezaza, `fin_` combined.

## 1. Setup

In [ ]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
YEARS = (2000, 2023)          # contract window
BUDGET_FLOOR = 1_000_000      # contract floor

## 2. TMDB

All released, non-adult films (one row per id). Matching uses a wider year window (1998–2025) so films near the edges can still match; the contract window is applied after.

In [ ]:
t = pd.read_csv(RAW / "TMDB_movie_dataset_v11.csv", low_memory=False)
t["release_dt"] = pd.to_datetime(t["release_date"], errors="coerce")
t["year"] = t["release_dt"].dt.year
tmdb = t[t["status"].eq("Released") & ~t["adult"] & t["year"].between(1998, 2025)].drop_duplicates("id").copy()

# Sanity check: Matt's contract population is reproduced exactly
pop = tmdb[tmdb["year"].between(*YEARS) & tmdb["budget"].ge(BUDGET_FLOOR) & tmdb["revenue"].gt(0)]
assert len(pop) == 5766, len(pop)
print(f"TMDB candidates for matching: {len(tmdb):,}; contract population: {len(pop):,}")

## 3. Load the other sources

In [ ]:
tn = pd.read_csv(RAW / "the_numbers/Top Movies (Cleaned Data).csv", low_memory=False)
tn["year"] = pd.to_datetime(tn["Release Date"], errors="coerce").dt.year
tn = tn[tn["year"].between(*YEARS)].reset_index(drop=True)

wd = pd.read_csv(RAW / "wikidata/wikidata_film_financials_2026-11-21.csv")
wd = wd.rename(columns={"tmdb": "id"}).drop_duplicates("id")

bom_daily = pd.read_csv(RAW / "boxofficemojo/revenues_per_day.csv.gz", parse_dates=["date"])
# `id` in this file is per ROW (film-day), not per film: collapse to one row per title
bom = (bom_daily.groupby("title")
       .agg(first_date=("date", "min"), domestic_gross=("revenue", "sum"),
            days=("date", "nunique"), max_theaters=("theaters", "max"), distributor=("distributor", "first"))
       .reset_index())
bom["year"] = bom["first_date"].dt.year

sl = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_movies.csv", low_memory=False)
sl["year"] = pd.to_datetime(sl["original_release_date"], errors="coerce").dt.year

pd.Series({"The Numbers (2000-23)": len(tn), "Wikidata": len(wd), "Box Office Mojo titles": len(bom),
           "RT stefanoleone992": len(sl)}, name="rows").to_frame()

## 4. Title + year matcher

1. **Normalize titles:** strip accents, lowercase, `&` → `and`, drop a leading *The / A / An*, keep letters and digits only.
2. **Candidates:** same normalized title as TMDB's `title` **or** `original_title`, release year within ±2.
3. **Pick the best candidate** per source film: budget agrees (±15%), then closest year, then closest runtime, then most TMDB votes (votes only break ties for identity; they're never a model input).
4. **Year differences need corroboration:** if years differ, runtime must be within 10 minutes or the budget must agree.
5. **One-to-one only:** a TMDB film claimed by two source films is dropped from both.

In [ ]:
def norm_title(s: pd.Series) -> pd.Series:
    s = s.fillna("").astype(str).map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode())
    s = s.str.lower().str.replace("&", " and ", regex=False)
    s = s.str.replace(r"^(the|a|an)\s+", "", regex=True)
    return s.str.replace(r"[^a-z0-9]", "", regex=True)


tmdb_keys = pd.concat([
    tmdb.assign(key=norm_title(tmdb["title"])),
    tmdb.assign(key=norm_title(tmdb["original_title"])),
]).drop_duplicates(["id", "key"])
tmdb_keys = tmdb_keys.loc[tmdb_keys["key"].ne(""), ["id", "key", "year", "runtime", "budget", "vote_count"]]


def match_to_tmdb(src, title_col, runtime_col=None, budget_col=None, name="src"):
    """Return src_row -> TMDB id, one-to-one, with match diagnostics."""
    s = pd.DataFrame({"src_row": np.arange(len(src)), "key": norm_title(src[title_col]), "src_year": src["year"]})
    s["src_runtime"] = src[runtime_col] if runtime_col else np.nan
    s["src_budget"] = src[budget_col] if budget_col else np.nan
    m = s.merge(tmdb_keys, on="key")
    m["year_diff"] = (m["src_year"] - m["year"]).abs()
    m = m[m["year_diff"] <= 2]
    m["runtime_diff"] = (m["src_runtime"] - m["runtime"]).abs()
    m["budget_agrees"] = m["budget"].gt(0) & ((m["budget"] / m["src_budget"] - 1).abs() < 0.15)
    m = (m.sort_values(["src_row", "budget_agrees", "year_diff", "runtime_diff", "vote_count"],
                       ascending=[True, False, True, True, False], na_position="last")
          .drop_duplicates("src_row"))
    m = m[m["year_diff"].eq(0) | m["runtime_diff"].le(10) | m["budget_agrees"]]
    m = m[~m["id"].duplicated(keep=False)]
    print(f"{name:<12} matched {len(m):,} of {len(src):,} ({len(m) / len(src):.0%}); "
          f"year diff counts {m['year_diff'].value_counts().sort_index().to_dict()}")
    return m[["src_row", "id", "year_diff", "runtime_diff", "budget_agrees"]]


tn_match = match_to_tmdb(tn, "Movie Name", "Running Time (minutes)", "Production Budget (USD)", "The Numbers")
bom_match = match_to_tmdb(bom, "title", name="Box Office Mojo")
sl_match = match_to_tmdb(sl, "movie_title", "runtime", name="RT sl")

both = tn_match.merge(tmdb[["id", "budget"]], on="id")
both = both[both["budget"].gt(0)]
print(f"\nCheck: where TMDB and The Numbers both have a budget, they agree (±15%) for "
      f"{both['budget_agrees'].mean():.0%} of {len(both):,} matched films")

## 5. Attach the sources to TMDB

In [ ]:
def attach(base, src, match, prefix, cols):
    part = src.iloc[match["src_row"]][cols].set_axis(match["id"].values)
    part.columns = [f"{prefix}_{c}" for c in cols]
    part[f"{prefix}_year_diff"] = match["year_diff"].values
    return base.merge(part, left_on="id", right_index=True, how="left")


tn_cols = ["Movie Name", "Release Date", "Production Budget (USD)", "Worldwide Gross (USD)", "Domestic Gross (USD)",
           "MPAA Rating", "Source", "Creative Type", "Production Method", "Franchise",
           "Production/Financing Companies", "Running Time (minutes)", "Movie URL"]
films = tmdb.copy()
films = attach(films, tn, tn_match, "tn", tn_cols)
films = attach(films, bom, bom_match, "bom", ["title", "first_date", "domestic_gross", "max_theaters"])
films = attach(films, sl, sl_match, "sl", ["rotten_tomatoes_link", "movie_title", "actors", "production_company", "directors"])
films = films.merge(wd.add_prefix("wd_").rename(columns={"wd_id": "id"}), on="id", how="left")
assert films["id"].is_unique

## 6. Combined budget and revenue

First non-missing value wins: TMDB → The Numbers → Wikidata. Zero means missing in every source.

In [ ]:
def first_valid(*cols):
    vals = pd.concat([c.where(c > 0) for c, _ in cols], axis=1)
    src = pd.Series(pd.NA, index=vals.index, dtype="string")
    for i, (_, label) in reversed(list(enumerate(cols))):
        src = src.mask(vals.iloc[:, i].notna(), label)
    return vals.bfill(axis=1).iloc[:, 0], src


films["fin_budget"], films["fin_budget_src"] = first_valid(
    (films["budget"], "tmdb"), (films["tn_Production Budget (USD)"], "the_numbers"), (films["wd_wd_budget_usd"], "wikidata"))
films["fin_revenue"], films["fin_revenue_src"] = first_valid(
    (films["revenue"], "tmdb"), (films["tn_Worldwide Gross (USD)"], "the_numbers"), (films["wd_wd_box_office_usd"], "wikidata"))

films["usable"] = (films["year"].between(*YEARS) & films["fin_budget"].ge(BUDGET_FLOOR) & films["fin_revenue"].gt(0))
usable = films[films["usable"]]
print(f"Usable films: {len(usable):,} (TMDB alone: {len(pop):,}, +{len(usable) - len(pop):,})")
pd.crosstab(usable["fin_budget_src"], usable["fin_revenue_src"], margins=True)

## 7. RT movies (andrezaza): director, writer, RT box office

Uses **Matt's join rules exactly** (`Sorensen-A10.ipynb` §4, hand-checked 0/50 wrong): normalized title (`title`, then `original_title`), RT year within ±1, or rule 2b (streaming-only RT date later than release, runtime within 5 min); best candidate = title before original_title, rule 2 before 2b, closest year, closest runtime; one-to-one, ambiguous pairs dropped.

Matt ran it on the 5,766-film population. Here it runs on every 2000–2023 film with any financial value, after first checking that the same code reproduces his 4,673 pairs.

In [ ]:
def matt_norm(s):
    s = s.fillna("").astype(str).map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode())
    s = s.str.lower().str.replace("&", " and ", regex=False)
    s = s.str.replace(r"[^a-z0-9 ]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def parse_money(s):
    m = s.str.extract(r"^\$([\d.]+)([KMB]?)$")
    return m[0].astype(float) * m[1].map({"": 1, "K": 1e3, "M": 1e6, "B": 1e9})


rt_movies = pd.read_csv(RAW / "rotten_tomatoes_movies.csv", low_memory=False)
rtj = rt_movies.drop_duplicates("id").copy()
rt_theat = pd.to_datetime(rtj["releaseDateTheaters"], errors="coerce").dt.year
rt_stream = pd.to_datetime(rtj["releaseDateStreaming"], errors="coerce").dt.year
rtj["rt_year"] = rt_theat.fillna(rt_stream)
rtj["rt_year_source"] = np.where(rt_theat.notna(), "theatrical", np.where(rt_stream.notna(), "streaming", "none"))
rtj["key"] = matt_norm(rtj["title"])
rtj = rtj[rtj["key"].ne("") & rtj["rt_year"].notna()]


def rt_join(tj):
    """Matt's §4 rules. tj needs id, title, original_title, year, runtime. Returns the 1:1 crosswalk."""
    tj = tj.assign(key_title=matt_norm(tj["title"]), key_orig=matt_norm(tj["original_title"]))
    side = rtj[["id", "key", "rt_year", "rt_year_source", "runtimeMinutes"]].rename(columns={"id": "rt_id"})
    cands = []
    for key_col, how in [("key_title", "title"), ("key_orig", "original_title")]:
        c = tj[["id", "year", "runtime", key_col]].merge(side, left_on=key_col, right_on="key")
        cands.append(c.assign(matched_on=how))
    cands = pd.concat(cands, ignore_index=True)
    signed = cands["rt_year"] - cands["year"]
    cands["year_diff"] = signed.abs()
    cands["runtime_diff"] = (cands["runtimeMinutes"] - cands["runtime"].replace(0, np.nan)).abs()
    rule2 = cands["year_diff"] <= 1
    rule2b = (cands["rt_year_source"] == "streaming") & (signed > 1) & (cands["runtime_diff"] <= 5)
    cands["rule"] = np.where(rule2, "2: year +/-1", np.where(rule2b, "2b: later streaming + runtime", None))
    cands = cands[rule2 | rule2b].copy()
    cands["title_rank"] = cands["matched_on"].map({"title": 0, "original_title": 1})
    cands["rule_rank"] = cands["rule"].str.startswith("2b").astype(int)
    cands = cands.sort_values(["id", "title_rank", "rule_rank", "year_diff", "runtime_diff"], na_position="last")
    best = cands.drop_duplicates("id")
    return best[~best["rt_id"].duplicated(keep=False)][["id", "rt_id", "matched_on", "rule", "year_diff", "runtime_diff"]]


# 1) Same code on Matt's population must give his 4,673 pairs
assert len(rt_join(pop)) == 4673, "doesn't reproduce Matt's crosswalk"

# 2) Two passes so the analysis films get first claim on each RT record:
#    usable films first, then the other 2000-2023 films against RT records not yet taken
scope = films["year"].between(*YEARS) & (films["fin_budget"].notna() | films["fin_revenue"].notna())
xw_usable = rt_join(films[films["usable"]])
taken = set(xw_usable["rt_id"])
xw_rest = rt_join(films[scope & ~films["usable"]])
xw_rest = xw_rest[~xw_rest["rt_id"].isin(taken)]
rt_xw = pd.concat([xw_usable, xw_rest], ignore_index=True)
assert rt_xw["id"].is_unique and rt_xw["rt_id"].is_unique

matt_pairs = rt_join(pop)
kept = xw_usable.merge(matt_pairs, on=["id", "rt_id"]).shape[0]
print(f"RT matches: {len(xw_usable):,} usable films ({len(xw_usable) / films['usable'].sum():.0%}) "
      f"+ {len(xw_rest):,} other films. Matt's verified pairs kept: {kept:,} of {len(matt_pairs):,}")

In [ ]:
rt_cols = [c for c in rt_movies.columns if c != "id"]
rt_part = (rt_xw.merge(rtj[["id"] + rt_cols].rename(columns={"id": "rt_id"}), on="rt_id")
               .assign(boxOffice_usd=lambda d: parse_money(d["boxOffice"])))
rt_part = rt_part.set_index("id").add_prefix("rt_").rename(columns={"rt_rt_id": "rt_id"})
films = films.drop(columns=[c for c in films.columns if c.startswith("rt_")]).merge(
    rt_part, left_on="id", right_index=True, how="left")
assert films["id"].is_unique
films.loc[films["usable"], ["rt_id", "rt_director", "rt_writer", "rt_boxOffice_usd"]].notna().mean().map("{:.0%}".format)

## 7b. Critic reviews, rolled up to one row per film

Two review files, each keyed by an RT slug, so they attach through the RT links already on each film (no title matching):

| File | Attaches via | Prefix |
|---|---|---|
| `rotten_tomatoes_movie_reviews.csv` (1.44M) | `rt_id` (RT andrezaza join) | `rtrev_` |
| `stefanoleone992/rotten_tomatoes_critic_reviews.csv` (1.13M) | `sl_rotten_tomatoes_link` | `slrev_` |

Per film: review count, top-critic count, % fresh, first and last review date, and how many reviews are dated **before** the TMDB release date (plan step 11, "early buzz").

⚠️ Reviews are known only after release, except the before-release counts. **They're descriptive columns, not model inputs** (contract: no post-release inputs). Review text is not copied (license: short excerpts only).

In [ ]:
def rollup(rev, key, fresh, top, date, prefix, release):
    """One row per film key: counts, % fresh, date range, reviews dated before release."""
    r = pd.DataFrame({"key": rev[key], "fresh": fresh, "top": top.astype(bool),
                      "date": pd.to_datetime(rev[date], errors="coerce")})
    r = r.merge(release, on="key", how="inner")
    r["before"] = r["date"] < r["release_dt"]
    g = r.groupby("key").agg(n_reviews=("fresh", "size"), n_top_critic=("top", "sum"),
                             pct_fresh=("fresh", "mean"), first_review=("date", "min"),
                             last_review=("date", "max"), n_before_release=("before", "sum"))
    g["pct_fresh"] = (100 * g["pct_fresh"]).round(1)
    return g.add_prefix(prefix)


# RT andrezaza reviews -> rt_id
rev_a = pd.read_csv(RAW / "rotten_tomatoes_movie_reviews.csv", low_memory=False,
                    usecols=["id", "creationDate", "isTopCritic", "reviewState"])
rel_a = films.loc[films["rt_id"].notna(), ["rt_id", "release_dt"]].rename(columns={"rt_id": "key"})
agg_a = rollup(rev_a, "id", rev_a["reviewState"].eq("fresh"), rev_a["isTopCritic"], "creationDate", "rtrev_", rel_a)

# RT stefanoleone992 reviews -> sl_rotten_tomatoes_link
rev_s = pd.read_csv(RAW / "stefanoleone992/rotten_tomatoes_critic_reviews.csv", low_memory=False,
                    usecols=["rotten_tomatoes_link", "review_type", "top_critic", "review_date"])
rel_s = (films.loc[films["sl_rotten_tomatoes_link"].notna(), ["sl_rotten_tomatoes_link", "release_dt"]]
              .rename(columns={"sl_rotten_tomatoes_link": "key"}))
agg_s = rollup(rev_s, "rotten_tomatoes_link", rev_s["review_type"].eq("Fresh"), rev_s["top_critic"],
               "review_date", "slrev_", rel_s)

films = (films.drop(columns=[c for c in films.columns if c.startswith(("rtrev_", "slrev_"))])
              .merge(agg_a, left_on="rt_id", right_index=True, how="left")
              .merge(agg_s, left_on="sl_rotten_tomatoes_link", right_index=True, how="left"))
assert films["id"].is_unique

u = films[films["usable"]]
pd.DataFrame({
    "usable films with reviews": [u["rtrev_n_reviews"].notna().mean(), u["slrev_n_reviews"].notna().mean()],
    "median reviews per film": [u["rtrev_n_reviews"].median(), u["slrev_n_reviews"].median()],
    "films with any review before release": [(u["rtrev_n_before_release"] > 0).mean(), (u["slrev_n_before_release"] > 0).mean()],
}, index=["RT andrezaza (rtrev_)", "RT stefanoleone992 (slrev_)"]).style.format(
    {"usable films with reviews": "{:.0%}", "median reviews per film": "{:.0f}",
     "films with any review before release": "{:.0%}"})

## 8. The Numbers films with no TMDB match

They have budget and gross but no TMDB fields (genre, language, etc.). Kept separately for now; whether to use them depends on which inputs the model needs.

In [ ]:
tn_only = tn.drop(index=tn_match["src_row"])
tn_only = tn_only[tn_only["Production Budget (USD)"].ge(BUDGET_FLOOR) & tn_only["Worldwide Gross (USD)"].gt(0)]
print(f"The Numbers films not in TMDB, usable on financials: {len(tn_only):,}")
print(f"Potential total: {len(usable) + len(tn_only):,}")

## 9. Hand-check sample

50 random The Numbers ↔ TMDB matches, weighted toward the riskier ones (year differs or budgets disagree). Review in `data/clean/edwards_tn_handcheck.csv`; mark `correct` as Y/N.

In [ ]:
rng = np.random.default_rng(560)
chk = tn_match.merge(tmdb[["id", "title", "original_title", "year", "runtime", "budget"]], on="id")
chk = chk.join(tn[["Movie Name", "year", "Running Time (minutes)", "Production Budget (USD)"]]
               .add_prefix("tn_"), on="src_row")
risky = chk[chk["year_diff"].gt(0) | ~chk["budget_agrees"]]
sample = pd.concat([risky.sample(25, random_state=rng.integers(1e9)),
                    chk.drop(risky.index).sample(25, random_state=rng.integers(1e9))])
sample["risky"] = sample.index.isin(risky.index)
sample["correct"] = ""
out = CLEAN / "edwards_tn_handcheck.csv"
if not out.exists():                      # written once, never overwritten (same rule as Matt's hand-check)
    sample.drop(columns="src_row").to_csv(out, index=False)
print(f"{len(risky):,} risky matches of {len(chk):,}; sample at {out}")

### 9b. Hand-check result

Reviewed by Tanner, 2026-11-22.

In [ ]:
from scipy.stats import beta

hc = pd.read_csv(CLEAN / "edwards_tn_handcheck.csv")
hc["ok"] = hc["correct"].astype(str).str.strip().str.upper().str.startswith("Y")
assert hc["correct"].notna().all(), "hand-check not finished"

def upper95(wrong, n):          # Clopper-Pearson exact 95% upper bound
    return beta.ppf(0.975, wrong + 1, n - wrong) if wrong < n else 1.0

res = hc.groupby("risky")["ok"].agg(checked="size", correct="sum")
res.loc["all"] = res.sum()
res["wrong"] = res["checked"] - res["correct"]
res["error rate 95% CI upper"] = [f"{upper95(w, n):.1%}" for w, n in zip(res["wrong"], res["checked"])]
res

## 10. Save the unified dataset

In [ ]:
# One row per film: 2000-2023 TMDB films with at least one budget or revenue value from any source.
# Fixed file names: every other notebook reads these instead of re-merging. Gitignored; rebuilt on every run.
unified = films[films["year"].between(*YEARS) & (films["fin_budget"].notna() | films["fin_revenue"].notna())]
assert unified["id"].is_unique
unified.to_csv(CLEAN / "films_unified.csv", index=False)
tn_only.to_csv(CLEAN / "films_tn_only.csv", index=False)
print(f"films_unified.csv: {len(unified):,} films x {unified.shape[1]} columns ({unified['usable'].sum():,} usable)")
print(f"films_tn_only.csv: {len(tn_only):,} The Numbers films with no TMDB match")